# Silver: Lojas (Centros de Distribuição)

**Objetivo:** transformar a tabela Bronze de Lojas em uma tabela Silver confiável e versionada, aplicando:
1. Padronização de texto: remoção de acentos e conversão para maiúsculas em `nome_cidade` (chave de junção usada por outras tabelas, ex: `dim_representantes`)
2. Controle SCD Tipo 2 real, via MERGE

**Fonte:** tabela Delta `bronze.lojas`.

**Destino:** tabela Delta `silver.lojas`.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, upper, translate, current_date, lit
from delta.tables import DeltaTable

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.silver")

print("Schema 'silver' verificado/criado com sucesso.")

In [0]:
# Padronização de texto - remoção de acentos, maiúsculas, e casting de peso_distribuicao

df_bronze_lojas = spark.table("poc_latam_food.bronze.lojas")

df_lojas_padronizado = (
    df_bronze_lojas
    .withColumn("nome_cidade", upper(translate(col("nome_cidade"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("pais", upper(translate(col("pais"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("peso_distribuicao", col("peso_distribuicao").cast("double"))
    .select("loja_id", "nome_cidade", "pais", "peso_distribuicao")
)

df_lojas_padronizado.display()


In [0]:
# SCD2 real via MERGE

tabela_existe = spark.catalog.tableExists("poc_latam_food.silver.lojas")

if not tabela_existe:
    df_primeira_carga = (
        df_lojas_padronizado
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )
    df_primeira_carga.write.format("delta").saveAsTable("poc_latam_food.silver.lojas")
    print("Carga inicial da Silver de Lojas concluída.")

else:
    tabela_silver = DeltaTable.forName(spark, "poc_latam_food.silver.lojas")

    colunas_comparadas = ["nome_cidade", "pais", "peso_distribuicao"]
    condicao_mudanca = " OR ".join([
        f"target.{c} <> source.{c}" for c in colunas_comparadas
    ])

    (
        tabela_silver.alias("target")
        .merge(
            df_lojas_padronizado.alias("source"),
            "target.loja_id = source.loja_id AND target.flag_ativo = true"
        )
        .whenMatchedUpdate(
            condition=condicao_mudanca,
            set={"flag_ativo": lit(False), "data_fim": current_date()}
        )
        .execute()
    )

    df_vigentes = spark.table("poc_latam_food.silver.lojas").filter("flag_ativo = true")

    df_para_inserir = (
        df_lojas_padronizado.alias("source")
        .join(df_vigentes.alias("target"), on="loja_id", how="left_anti")
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )

    df_para_inserir.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.lojas")

    print(f"MERGE concluído: {df_para_inserir.count()} nova(s) versão(ões) inserida(s).")

In [0]:
# Validação visual - silver.lojas

spark.table("poc_latam_food.silver.lojas").display()

print(f"Total de lojas: {spark.table('poc_latam_food.silver.lojas').count()}")